In [17]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Data Preparation

In [18]:
# load training dataset
train_path = "data/basic_dataset.csv"
chunk = pd.read_csv(train_path, chunksize=1000000)
df = pd.concat(chunk)

In [19]:
# replace relevance 5 label with 2
df['relevance'] = df['relevance'].replace(5, 2)

# split data based on search ids
train_groups, val_groups = train_test_split(df[['srch_id']].drop_duplicates(), test_size=0.05, random_state=42)

# split the actual data on those groups
train_data = df[df['srch_id'].isin(train_groups['srch_id'])]
val_data = df[df['srch_id'].isin(val_groups['srch_id'])]

# prepare X and y for train/val set
X_train = train_data.drop(columns=['click_bool', 'booking_bool', 'relevance', 'position', 'gross_bookings_usd'])
y_train = train_data['relevance']

X_val = val_data.drop(columns=['click_bool', 'booking_bool', 'relevance', 'position', 'gross_bookings_usd'])
y_val = val_data['relevance']


In [20]:
class Dataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [25]:
dataloader_train = DataLoader(Dataset(X_train.values.astype(float),y_train.values.astype(int)), batch_size=32, shuffle=True)
dataloader_val = DataLoader(Dataset(X_val.values.astype(float),y_val.values.astype(int)), batch_size=32, shuffle=True)

In [22]:
input_size = X_train.values.shape[1]

# Set up model

In [23]:
class RankNet(nn.Module):
    def __init__(self):
        super(RankNet, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 10), # input layer
            nn.ReLU(),
            nn.Linear(10, 3), # output layer
        )

    def forward(self, x):
        activated = self.layers(x)
        return activated

# Train model

In [26]:
# init model
model = RankNet()

# init criterion & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50

for epoch in tqdm(range(num_epochs),"epochs"):
    for features, labels in tqdm(dataloader_train, "train set"):
        # train mode on
        model.train()
        
        optimizer.zero_grad()
        out = model(features)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss = loss.item()
    
        
    # validation
    model.eval()
    val_loss = 0
    for features_val, labels_val in tqdm(dataloader_val,"validation"):
        outputs = model(features_val)
        val_loss += criterion(outputs.squeeze(), labels_val).item()
    val_loss /= len(dataloader_val)
    train_loss /= len(dataloader_train)
    
    print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, validation loss: {val_loss.item():.4f}')

epochs:   0%|          | 0/50 [03:31<?, ?it/s]


KeyboardInterrupt: 